> **⚠ SUPERSEDED FRAMING (2026-06-28).** This notebook is a point-in-time record (Notebook 15 — DINNDeep Architecture Upgrade Test on Eq Pacific FeT (Track 1 v1.4)); its framing predates the surrogate-to-model **identifiability study** that is now canonical (four observable parameters {`alpfe`, `scav_rat`, `diatomgraz`, `R_PICPOC`}; the growth pair {`Smallgrow`, `Biggrow`} is unobservable by construction; `R_PICPOC` is recoverable via a real calcite anchor; per-cell prediction is load-bearing for the {`alpfe`, `scav_rat`, `R_PICPOC`} trio, 7/10 vs 0/10 global-scalar). The 0-D surrogate gap is dimensional (the box homogenizes spatial structure). Treat its numbers and claims as historical. Canonical results: `STATUS.md` / `README.md`.

# Notebook 15 — DINNDeep Architecture Upgrade Test on Eq Pacific FeT (Track 1 v1.4)

**Goal.** Test whether a deeper, wider, multi-channel-input network beats the SST-only DINN on the project's weakest fit (nb14: Eq Pacific FeT, DINN r = 0.337). If yes, the upgraded architecture becomes the default for production fits going forward.

**Hypothesis.** Two factors are jointly limiting nb14's iron r:
1. **Single-covariate input** (SST only) is iron-blind — Eq Pacific iron is upwelling- and dust-driven, not SST-driven.
2. **Tiny network** (~454 weights) lacks capacity to represent the iron pattern even with the right inputs.

**Experiment design.** Same loss target (Darwin v05 FeT, time-mean climatology, bin-averaged to 1° lat/lon, Eq Pacific AOI) and same hyperparameters as nb14. Only differences:

| | nb14 baseline | **nb15 upgrade** |
|---|---|---|
| Network | DINN (1×1 conv, 2 layers, 16 dim, Tanh) | **DINNDeep (4 res blocks, 32 dim, GELU, per-cell LayerNorm)** |
| Input channels | SST only | **SST + MLD + windSpeed + latitude** (4 channels) |
| Param count | ~454 weights | ~9,400 weights (~21× bigger) |
| Training | Adam lr=5e-3, 1500 epochs | same |

Architecture is still **per-cell** (1×1 conv backbone, no spatial coupling) so the structural-ceiling argument is preserved. Only changes: more capacity + more inputs.

**Expected outcome.** If r improves substantially (e.g., 0.337 → 0.5+), the architecture upgrade is justified and we adopt DINNDeep + multi-channel as the new default. If r barely moves, the bottleneck is something else (box-model proxy bias, observation type, etc.).

**Stacked PR note.** This notebook depends on the new `DINNDeep` class added to `src/darwindiff/networks.py` in this PR's commit. Reuses the LLC270 loader from prior PRs already merged to main.

In [ ]:
import sys
import time
import warnings
from pathlib import Path

_repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
_src = _repo_root / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

warnings.filterwarnings("ignore", message="Couldn't find available_diagnostics.log")
warnings.filterwarnings("ignore", category=FutureWarning)

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.stats import binned_statistic_2d

from darwindiff.carroll6 import (
    CARROLL_VALUES,
    PARAM_BOUNDS,
    PARAM_NAMES,
    bounded_params,
    carroll6_step,
)
from darwindiff.diagnostics import format_pearson, safe_pearson_r
from darwindiff.ecco_darwin_loader import (
    EQUATORIAL_PACIFIC_AOI,
    open_bin_average,
    subset_aoi,
    time_mean,
)
from darwindiff.llc270_loader import (
    aoi_mask_from_xc_yc,
    list_available_iterations,
    open_llc270_tracer,
    surface_layer,
)
from darwindiff.networks import DINN, DINNDeep

torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__}, GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no'}")

## 1. Load FeT (target) + 4-channel covariates from bin_average

Same data path as nb14 for FeT. New: pull MLD, windSpeed, and build a latitude channel from the AOI grid.

In [ ]:
# === Data root (env-var driven for cluster portability; default keeps local behaviour) ===
import os
from pathlib import Path
DATA_ROOT = Path(os.environ.get("DARWIN_DATA_ROOT", r"D:\ecco_darwin_v5"))

MONTHLY_ROOT = str(DATA_ROOT / "output" / "monthly")
GRID_DIR = str(DATA_ROOT / "grid")
BIN_AVG_PATH = str(DATA_ROOT / "bin_average" / "v05_ECCO-Darwin_bin_average_1x1_deg.nc")
AOI = EQUATORIAL_PACIFIC_AOI

# === FeT target (same as nb14) ===
iters = list_available_iterations(MONTHLY_ROOT, "FeT")
fet_ds = open_llc270_tracer(MONTHLY_ROOT, GRID_DIR, "FeT", iters=iters)
fet_surf = surface_layer(fet_ds)
fet_mean = fet_surf.FeT.mean(dim="time", skipna=True).values  # (face, j, i)
xc_native = fet_surf.XC.values; yc_native = fet_surf.YC.values
good = aoi_mask_from_xc_yc(xc_native, yc_native, AOI.lat_min, AOI.lat_max, AOI.lon_min, AOI.lon_max) & np.isfinite(fet_mean) & (fet_mean != 0)
lat_edges = np.arange(AOI.lat_min - 0.5, AOI.lat_max + 0.5 + 0.001, 1.0)
lon_edges = np.arange(AOI.lon_min - 0.5, AOI.lon_max + 0.5 + 0.001, 1.0)
fet_binned, _, _, _ = binned_statistic_2d(yc_native[good], xc_native[good], fet_mean[good], statistic="mean", bins=[lat_edges, lon_edges])
print(f"FeT binned to 1deg: shape={fet_binned.shape}, finite={int(np.isfinite(fet_binned).sum())}")

# === Covariates from bin_average ===
ds_bin = open_bin_average(BIN_AVG_PATH)
eqpac_clim = time_mean(subset_aoi(ds_bin, AOI))
sst = eqpac_clim.SST.values
mld = eqpac_clim.mldDepth.values
wind = eqpac_clim.windSpeed.values
# Latitude channel: broadcast lat coord to the 2D field
lat_1d = eqpac_clim.lat.values
lat_2d = np.broadcast_to(lat_1d[:, None], sst.shape).astype(np.float64)

ocean_mask = np.isfinite(sst) & np.isfinite(mld) & np.isfinite(wind) & np.isfinite(fet_binned)
n_ocean = int(ocean_mask.sum())
print(f"Combined ocean cells (all 4 covariates + FeT present): {n_ocean} of {ocean_mask.size}")

for name, arr in [("SST", sst), ("MLD", mld), ("wind", wind), ("lat", lat_2d), ("FeT target", fet_binned)]:
    a = arr[ocean_mask]
    print(f"  {name:>11s}: range [{a.min():.3e}, {a.max():.3e}], mean {a.mean():.3e}, std {a.std():.3e}")

## 2. Build training tensors (SST-only + 4-channel variants) + z-scored target

We'll train two networks on the SAME data: DINN with SST-only (matches nb14 baseline) and DINNDeep with all 4 channels. Z-scored FeT target shared between them.

In [ ]:
def normalize(arr, mask):
    """Z-score over ocean cells, zero outside; returns float32 array."""
    ocean = arr[mask]
    return np.where(mask, (arr - ocean.mean()) / max(ocean.std(), 1e-9), 0.0).astype(np.float32)

sst_norm = normalize(sst, ocean_mask)
mld_norm = normalize(mld, ocean_mask)
wind_norm = normalize(wind, ocean_mask)
lat_norm = normalize(lat_2d, ocean_mask)

# SST-only (1 channel) for DINN baseline
env_1ch = torch.tensor(sst_norm, dtype=torch.float32).unsqueeze(0)
# 4-channel (SST, MLD, wind, lat) for DINNDeep
env_4ch = torch.tensor(np.stack([sst_norm, mld_norm, wind_norm, lat_norm], axis=0), dtype=torch.float32)

fet_clean = np.where(ocean_mask, fet_binned, 1.0)
fet_target = torch.tensor(fet_clean, dtype=torch.float32)
mask_t = torch.tensor(ocean_mask, dtype=torch.bool)
H, W = env_1ch.shape[1], env_1ch.shape[2]
state0 = torch.tensor([5.0e-4, 1.0, 1.0, 0.5, 0.025]).reshape(5, 1, 1).expand(5, H, W).contiguous()

env_1ch_dev = env_1ch.to(device); env_4ch_dev = env_4ch.to(device)
state0_dev = state0.to(device); fet_target_dev = fet_target.to(device); mask_dev = mask_t.to(device)
bounds_dev = PARAM_BOUNDS.to(device)

fet_ocean = fet_target_dev[mask_dev]
target_mean = fet_ocean.mean(); target_std = fet_ocean.std().clamp(min=1e-6)
target_z = (fet_target_dev - target_mean) / target_std

print(f"env_1ch (SST-only) shape: {tuple(env_1ch.shape)}")
print(f"env_4ch (SST+MLD+wind+lat) shape: {tuple(env_4ch.shape)}")
print(f"FeT target z-scored: ocean mean={float(target_mean):.3e}, std={float(target_std):.3e}")

## 3. Train both networks: SST-only DINN baseline + 4-channel DINNDeep

Same loss (z-scored DFe vs z-scored FeT), same hyperparameters (Adam lr=5e-3, 1500 epochs). Two training runs, ~14 min total on RTX 5090.

In [ ]:
DT, N_STEPS, N_EPOCHS = 0.25, 200, 1500

def train(net, env_dev, seed: int = 0) -> dict:
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(net.parameters(), lr=5e-3)
    losses = []
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for epoch in range(N_EPOCHS):
        optimizer.zero_grad()
        params = bounded_params(net(env_dev), bounds_dev)
        state = state0_dev
        for _ in range(N_STEPS):
            state = carroll6_step(state, params, DT)
        dfe = state[0]
        dfe_ocean = dfe[mask_dev]
        dfe_z = (dfe - dfe_ocean.mean()) / dfe_ocean.std().clamp(min=1e-6)
        residual = (dfe_z - target_z) * mask_dev.to(dfe.dtype)
        loss = (residual ** 2).sum() / mask_dev.sum().to(residual.dtype)
        loss.backward(); optimizer.step()
        losses.append(loss.item())
        if (epoch + 1) % 250 == 0:
            print(f"    epoch {epoch+1:4d}  loss = {loss.item():.4e}")
    if device == "cuda": torch.cuda.synchronize()
    elapsed = time.time() - t0
    with torch.no_grad():
        params_final = bounded_params(net(env_dev), bounds_dev).cpu()
        state = state0_dev
        for _ in range(N_STEPS):
            state = carroll6_step(state, bounded_params(net(env_dev), bounds_dev), DT)
        dfe_final = state[0].cpu()
    return {"losses": losses, "params_final": params_final, "dfe_final": dfe_final, "elapsed": elapsed}

# Baseline: DINN with SST-only input (matches nb14)
torch.manual_seed(0)
dinn_baseline = DINN(n_input_channels=1, hidden_dim=16, n_outputs=6).to(device)
n_b = sum(p.numel() for p in dinn_baseline.parameters())
print(f"=== DINN baseline (SST-only, {n_b} params) ===")
r_baseline = train(dinn_baseline, env_1ch_dev)
print(f"  done in {r_baseline['elapsed']:.0f}s, loss {r_baseline['losses'][0]:.3e} -> {r_baseline['losses'][-1]:.3e}")

# Upgrade: DINNDeep with 4-channel input
torch.manual_seed(0)
dinn_deep = DINNDeep(n_input_channels=4, hidden_dim=32, n_outputs=6, n_blocks=4).to(device)
n_d = sum(p.numel() for p in dinn_deep.parameters())
print(f"\n=== DINNDeep (SST+MLD+wind+lat, {n_d} params) ===")
r_deep = train(dinn_deep, env_4ch_dev)
print(f"  done in {r_deep['elapsed']:.0f}s, loss {r_deep['losses'][0]:.3e} -> {r_deep['losses'][-1]:.3e}")

## 4. Pearson r + recovered iron-pair comparison

In [ ]:
for r in [r_baseline, r_deep]:
    assert torch.isfinite(r["dfe_final"][mask_t]).all(), "DFe integration produced NaN"

n_total = int(ocean_mask.sum())
target_raw = fet_binned[ocean_mask]
result_b = safe_pearson_r(r_baseline["dfe_final"].numpy()[ocean_mask], target_raw)
result_d = safe_pearson_r(r_deep["dfe_final"].numpy()[ocean_mask], target_raw)

print("Pearson correlation, predicted DFe vs Darwin FeT (Eq Pacific HNLC):")
print(f"  DINN baseline   (SST only,    {n_b:>5} params):  r = {format_pearson(result_b, n_total=n_total)}")
print(f"  DINNDeep        (4-ch input,  {n_d:>5} params):  r = {format_pearson(result_d, n_total=n_total)}")
print()
print(f"Loss plateau:")
print(f"  DINN baseline:  {r_baseline['losses'][-1]:.4f}")
print(f"  DINNDeep:       {r_deep['losses'][-1]:.4f}")
improvement = (r_baseline['losses'][-1] - r_deep['losses'][-1]) / r_baseline['losses'][-1] * 100
print(f"  Loss reduction: {improvement:.1f}%")

print("\nRecovered Carroll-6 (focus on iron pair):")
print(f"  {'param':<11s} {'DINN baseline mean':>20s} {'DINNDeep mean':>16s} {'Carroll published':>17s}")
for i, name in enumerate(PARAM_NAMES):
    p_b = r_baseline["params_final"][i].numpy()[ocean_mask]
    p_d = r_deep["params_final"][i].numpy()[ocean_mask]
    pub = float(CARROLL_VALUES[i])
    star = "  <- iron pair" if name in {"alpfe", "scav_rat"} else ""
    print(f"  {name:<11s} {p_b.mean():>20.4e} {p_d.mean():>16.4e} {pub:>17.4e}{star}")

## 5. Plots — target FeT, baseline DINN prediction, DINNDeep prediction, loss curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fet_plot = np.where(ocean_mask, fet_binned, np.nan)
dfe_b = np.where(ocean_mask, r_baseline["dfe_final"].numpy(), np.nan)
dfe_d = np.where(ocean_mask, r_deep["dfe_final"].numpy(), np.nan)

im0 = axes[0, 0].imshow(fet_plot, origin="lower", aspect="auto", cmap="viridis")
axes[0, 0].set_title("Darwin FeT target (Eq Pacific)")
plt.colorbar(im0, ax=axes[0, 0])

im1 = axes[0, 1].imshow(dfe_b, origin="lower", aspect="auto", cmap="plasma")
axes[0, 1].set_title(f"DINN baseline (SST-only)\n({format_pearson(result_b)[:30]})")
plt.colorbar(im1, ax=axes[0, 1])

im2 = axes[1, 0].imshow(dfe_d, origin="lower", aspect="auto", cmap="plasma")
axes[1, 0].set_title(f"DINNDeep (4-channel)\n({format_pearson(result_d)[:30]})")
plt.colorbar(im2, ax=axes[1, 0])

axes[1, 1].semilogy(r_baseline["losses"], label=f"DINN baseline ({n_b} params)", color="tab:red")
axes[1, 1].semilogy(r_deep["losses"], label=f"DINNDeep ({n_d} params)", color="tab:green")
axes[1, 1].set_title("Loss curves"); axes[1, 1].set_xlabel("epoch"); axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## What this notebook tests — and what to do with the result

**The experiment is binary:** does multi-channel DINNDeep substantially improve r over the SST-only DINN baseline on the project's weakest fit?

- **If yes (e.g., r jumps from 0.337 to 0.5+):** adopt DINNDeep + multi-channel as the new default architecture. Re-run nb11 / nb13 with the same upgrade for a unified "v2" set of fits across all (AOI × target) combinations.
- **If small improvement only (e.g., 0.337 → 0.4):** the bottleneck is partly architecture but likely also box-model proxy bias. Multi-tracer joint loss (nb16+) and box-model carbonate-chemistry extension become higher priority than further architecture work.
- **If no improvement:** the iron pattern in Eq Pacific is fundamentally hard to fit with any per-cell SST/MLD/wind-conditioned network. May need spatial coupling (different scientific question, see receptive-field discussion in project memory) or different observations (depth-resolved iron, GEOTRACES sections).

**What stays the same regardless of outcome:**
- Per-cell architecture (1×1 conv backbone, no spatial coupling)
- Sigmoid bounding into Carroll's PARAM_BOUNDS
- Box model = carroll6 (5 tracers, no carbonate chem)
- Loss = z-scored MSE on the structural target
- Structural-ceiling argument (DINN class beats global-scalar Green's-functions class)

## Where this fits in the project arc

- 09–14: SST-only DINN fits across multiple (AOI × target) combos
- **15 (this notebook): architecture upgrade test on the weakest fit** — Track 1 v1.4
- 16+: outcome-dependent. If DINNDeep wins, broad re-run; if not, multi-tracer / box-model extension